# 5 — Dummy baseline


## Imports


In [1]:
from pathlib import Path
import hashlib
import importlib.metadata
import json
import platform
import gc
import random
from time import perf_counter
import joblib
import numpy as np
import pandas as pd
import torch
from sklearn.dummy import DummyClassifier
from sklearn.metrics import f1_score


## Configuration


In [2]:
from pathlib import Path
# ใช้ project root เดียวกัน ไม่ต้อง import โค้ด pipeline จาก program
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / 'dataset').is_dir() and (p / 'model').is_dir() and (p / 'program').is_dir())
RAW = ROOT / 'dataset/raw/mit-bih-arrhythmia-database-1.0.0'
DATA = ROOT / 'dataset/processed'
ARTIFACTS = ROOT / 'model/artifacts'
REPORTS = ROOT / 'model/reports'
CLASSES = ['N', 'S', 'V', 'F', 'Q']
SYMBOLS = ['NLRej', 'AaJS', 'VE', 'F', '/fQ']
LABEL_MAP = {s: i for i, group in enumerate(SYMBOLS) for s in group}
SEED = 42
PREPROCESSING = {'version': 1, 'fs': 360, 'left': 90, 'right': 162,
                 'band': [0.5, 40], 'order': 4, 'leads': ['MLII', 'V5']}
MODEL = 'dummy'
TRIALS = 0
BASELINE_PARAMETERS = {}

## Artifact helpers


In [3]:
def read_json(path):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f'ไม่พบ {path}; กรุณารัน notebook ขั้นก่อนหน้าก่อน')
    return json.loads(path.read_text(encoding='utf-8'))

def write_json(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(value, ensure_ascii=False, indent=2, allow_nan=False), encoding='utf-8')

def sha256(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()

def new_stage(marker):
    # ห้ามเขียนทับผลเดิมเงียบ ๆ เพราะอาจทำให้โมเดลอ้างอิงคนละ split
    if Path(marker).exists():
        raise FileExistsError(f'มีผลขั้นนี้แล้ว: {marker}; โหลดผลเดิมแทนการรันซ้ำ')
    if (ARTIFACTS / 'selection.json').exists():
        raise RuntimeError('ล็อกโมเดลแล้ว ไม่อนุญาตให้เตรียมข้อมูลหรือ tuning ใหม่ใน experiment เดิม')

## Load partition helper


In [4]:
def load_partition(split, *, allow_test=False):
    if split not in ['train', 'validation', 'test']:
        raise ValueError('Unknown partition')
    if split == 'test' and not allow_test:
        raise PermissionError('Test เปิดได้เฉพาะ final evaluation หลังล็อกโมเดล')
    manifest = read_json(DATA / 'prepared.json')
    folder = DATA / split
    for name in ['metadata.parquet', 'waveforms.npy', 'features.npy']:
        if sha256(folder / name) != manifest['files'][f'{split}/{name}']:
            raise ValueError(f'artifact เปลี่ยน: {split}/{name}')
    meta = pd.read_parquet(folder / 'metadata.parquet')
    waves, x = np.load(folder / 'waveforms.npy'), np.load(folder / 'features.npy')
    if len(meta) != len(waves) or len(meta) != len(x) or meta.beat_id.duplicated().any():
        raise ValueError('beat alignment ไม่ถูกต้อง')
    if not meta.split.eq(split).all() or set(meta.class_id) != set(range(5)):
        raise ValueError('partition หรือ class coverage ไม่ถูกต้อง')
    if not np.array_equal(meta.waveform_index.to_numpy(), np.arange(len(meta))):
        raise ValueError('waveform index ไม่ตรงกับ metadata')
    if waves.shape != (len(meta), 252) or not np.isfinite(waves).all() or not np.isfinite(x).all():
        raise ValueError('ขนาด waveform หรือค่าข้อมูลหลัง preprocessing ไม่ถูกต้อง')
    return {'meta': meta, 'wave': waves, 'x': x, 'y': meta.class_id.to_numpy()}

## Metrics and training weights


In [5]:
def macro_f1(y, predicted):
    return float(f1_score(y, predicted, labels=list(range(5)), average='macro', zero_division=0))

def weights_from_train(y):
    counts = np.bincount(y, minlength=5)
    if (counts == 0).any():
        raise ValueError('Train ต้องมีครบ 5 คลาส')
    return len(y) / (5 * counts)

## Reproducibility


In [6]:
def environment():
    packages = ['numpy', 'pandas', 'scipy', 'scikit-learn', 'wfdb', 'xgboost',
                'torch', 'optuna', 'streamlit', 'pyarrow']
    versions = {}
    for package in packages:
        try:
            versions[package] = importlib.metadata.version(package)
        except importlib.metadata.PackageNotFoundError:
            versions[package] = 'not installed'
    return {'python': platform.python_version(), 'platform': platform.platform(),
            'processor': platform.processor(), 'packages': versions}

def set_seed():
    random.seed(SEED)
    np.random.seed(SEED)
    torch.manual_seed(SEED)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True

## Prediction


In [7]:
def probabilities(model, kind, values, device='cpu', batch_size=128):
    output = np.zeros((len(values), 5), dtype=np.float32)
    output[:, np.asarray(model.classes_, dtype=int)] = model.predict_proba(values)
    return output

def synchronize(device):
    if str(device).startswith('cuda'):
        torch.cuda.synchronize()

## Fit and save one candidate


In [8]:
# Train เท่านั้นที่ปรับน้ำหนักโมเดล; Validation ใช้ให้คะแนนและเลือก checkpoint
def fit_candidate(kind, weighted, parameters, folder, train, validation, *, epochs=60, rounds=1000):
    folder.mkdir(parents=True, exist_ok=False)
    set_seed()
    is_dl = False
    device = 'cuda' if is_dl and torch.cuda.is_available() else 'cpu'
    if is_dl and device != 'cuda':
        raise RuntimeError('แผนนี้ใช้ GPU สำหรับ DL: ติดตั้ง PyTorch CUDA ก่อนรัน')
    weights = weights_from_train(train['y']) if weighted else None
    started = perf_counter()
    history, best_round, used_batch, oom_batches = ({}, None, None, [])
    model = DummyClassifier(strategy='most_frequent').fit(train['x'], train['y'])
    joblib.dump(model, folder / 'model.joblib')
    synchronize(device)
    train_seconds = perf_counter() - started
    values = validation['wave'] if is_dl else validation['x']
    proba = probabilities(model, kind, values, device)
    score = macro_f1(validation['y'], proba.argmax(1))
    result = validation['meta'].copy()
    result['predicted'] = proba.argmax(1)
    for i in range(5):
        result[f'score_{i}'] = proba[:, i]
    result.to_parquet(folder / 'validation_predictions.parquet', index=False)
    info = {'kind': kind, 'weighted': weighted, 'seed': SEED, 'parameters': parameters, 'validation_macro_f1': score, 'train_seconds': train_seconds, 'best_epoch_or_round': best_round, 'batch_size': used_batch, 'oom_batch_attempts': oom_batches, 'device': device, 'gpu': torch.cuda.get_device_name(0) if device == 'cuda' else None, 'class_weights': None if weights is None else weights.tolist(), 'prepared_hash': sha256(DATA / 'prepared.json'), 'environment': environment()}
    write_json(folder / 'info.json', info)
    del model
    gc.collect()
    if device == 'cuda':
        torch.cuda.empty_cache()
    return info

## Load Train and Validation


In [9]:
# ไม่อ่าน Test และไม่แบ่ง inner validation
new_stage(ARTIFACTS / 'selection.json')
train = load_partition('train')
validation = load_partition('validation')

## Baseline


In [10]:
# ฝึก baseline ก่อน เพื่อเทียบกับผล tuning โดยใช้ split เดียวกัน
runs = {}
for weighted in [False]:
    name = f'{MODEL}_{"weighted" if weighted else "unweighted"}'
    folder = ARTIFACTS / name
    new_stage(folder / 'selected.json')
    if folder.exists():
        raise FileExistsError(f'{folder} มีผลอยู่แล้ว ตรวจหรือเก็บสำรองก่อนเริ่มใหม่')
    folder.mkdir(parents=True)
    baseline = fit_candidate(MODEL, weighted, BASELINE_PARAMETERS, folder / 'baseline', train, validation)
    runs[name] = {'folder': folder, 'weighted': weighted, 'baseline': baseline,
                  'candidates': [('baseline', baseline)], 'tuning_seconds': 0.}

## Select and save candidates


In [11]:
# เลือกตัวแทนแต่ละ configuration จาก Validation เท่านั้น
selected_candidates = []
for name, run in runs.items():
    selected_id, info = max(run['candidates'], key=lambda pair: pair[1]['validation_macro_f1'])
    selected = {'configuration': name, 'candidate': selected_id,
                'validation_macro_f1': info['validation_macro_f1'],
                'baseline_macro_f1': run['baseline']['validation_macro_f1'],
                'tuning_seconds': run['tuning_seconds'], 'prepared_hash': info['prepared_hash']}
    write_json(run['folder'] / 'selected.json', selected)
    selected_candidates.append(selected)
pd.DataFrame(selected_candidates)

,configuration,candidate,validation_macro_f1,baseline_macro_f1,tuning_seconds,prepared_hash
0,dummy_unweighted,baseline,0.175314,0.175314,0.0,1039391cb37a162946ad9ed42d0cca3d0a5e6bc09fe458...
